# Week 3 — Data contract for refresh opportunity ranking

This notebook describes the Refresh / Content Opportunity Scoring lane using a real mid-panel slice of the approved FlyRank warehouse release. The final June sample is deliberately not used for development.

## 1) Contract in plain words

1. **What one row means:** one row in the daily fact table represents one content item for one client on one report date: `(report_date, client_hash_id, content_hash_id)`.
2. **Which table(s):** I use `fact_content_daily_performance`, specifically the `month=2026-03` partition. It contains the observable search and analytics signals needed for this first refresh-ranking slice.
3. **Which time window:** the fact slice is March 1–31, 2026. The decision moment is the end of that month for this framing exercise; June 2026 remains a sealed final-month test and is not used here.
4. **What I would predict or rank:** rank content items by likely refresh-review priority. The ideal future label would be `future_refresh_opportunity`, based on a later outcome such as sustained improvement in clicks or sessions after a documented action. It is not present here, so the notebook uses a clearly provisional current-window proxy only for the leakage demonstration.
5. **One deliberate exclusion:** I exclude identifiers from features, and I do not use product decisions such as `priority_score`, `action_type`, or `health_score`. They are not safe discovery features and are not in this observable release.

## 2) Access and lane choice

The selected lane is supported by the repository's existing refresh queue, baseline rules, and data guide. The March partition was downloaded through the authorized Hugging Face session and saved outside the repository at runtime. In a fresh clone, set `HF_TOKEN` as an environment variable or run the documented DuckDB `hf://` path; never paste a token into a cell.

In [1]:
from pathlib import Path
import os
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists()
)
MONTH = "2026-03"
cached_partition = REPO_ROOT.parent / "_flyrank_w03_march_data_0.parquet"

con = duckdb.connect()
if cached_partition.exists():
    fact_source = f"read_parquet('{cached_partition.as_posix()}')"
    source_note = "authorized local cache of the March 2026 warehouse partition"
elif os.environ.get("HF_TOKEN"):
    token = os.environ["HF_TOKEN"].replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")
    fact_source = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
    source_note = "FlyRank/internship-warehouse via DuckDB hf://"
else:
    raise RuntimeError("No March partition cache found. Set HF_TOKEN in the environment; do not paste it into this notebook.")

print("Source:", source_note)
print("Month under review:", MONTH)

Source: authorized local cache of the March 2026 warehouse partition
Month under review: 2026-03


## 3) Three verification queries

These are the only three verification queries. They check the grain, the March slice's row count/date span, and Analytics availability using an explicit `IS TRUE` filter.

In [2]:
# Verification query 1 — grain
grain_check = con.execute(f"""
    WITH per_grain AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_per_grain
        FROM {fact_source}
        GROUP BY report_date, client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS distinct_grain_keys,
        SUM(rows_per_grain) AS source_rows,
        MAX(rows_per_grain) AS max_rows_per_grain,
        MIN(rows_per_grain) AS min_rows_per_grain
    FROM per_grain
""").fetchdf()
display(grain_check)
print("Interpretation: max_rows_per_grain = 1 proves one daily fact row per report_date + client + content key.")

,distinct_grain_keys,source_rows,max_rows_per_grain,min_rows_per_grain
0,9841378,9841378.0,1,1


Interpretation: max_rows_per_grain = 1 proves one daily fact row per report_date + client + content key.


In [3]:
# Verification query 2 — slice row count and date span
slice_check = con.execute(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS report_days,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {fact_source}
""").fetchdf()
display(slice_check)

,march_rows,first_report_date,last_report_date,report_days,clients,content_items
0,9841378,2026-03-01,2026-03-31,31,55,331437


In [4]:
# Verification query 3 — availability, explicitly using IS TRUE
availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(DISTINCT content_hash_id) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_content_items
    FROM {fact_source}
    WHERE ga4_data_available IS TRUE
""").fetchdf()
display(availability_check)
print("The availability count is restricted with ga4_data_available IS TRUE; NULL and FALSE rows do not survive.")

,march_rows,ga4_available_rows,ga4_available_content_items
0,413966,413966,90489


The availability count is restricted with ga4_data_available IS TRUE; NULL and FALSE rows do not survive.


## 4) Five features and the deliberate leakage trap

The feature frame aggregates March at the content-item/client grain. These are five candidate features only:

- `march_gsc_impressions`: knowable at the decision moment because Search Console delivery for March has closed.
- `march_gsc_clicks`: knowable at the decision moment because it is a March aggregate already observed.
- `march_avg_position`: knowable at the decision moment because it is calculated from March search observations.
- `march_active_gsc_days`: knowable at the decision moment because it counts March days with impressions.
- `march_ga4_sessions`: knowable at the decision moment only when `ga4_data_available IS TRUE`; otherwise it must be treated as unavailable, not as a true zero.

In [5]:
feature_frame = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS march_gsc_impressions,
        SUM(COALESCE(gsc_clicks, 0)) AS march_gsc_clicks,
        AVG(gsc_avg_position) FILTER (WHERE gsc_data_available IS TRUE) AS march_avg_position,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS march_active_gsc_days,
        SUM(COALESCE(ga4_sessions, 0)) FILTER (WHERE ga4_data_available IS TRUE) AS march_ga4_sessions
    FROM {fact_source}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(COALESCE(gsc_impressions, 0)) > 0
""").fetchdf()

feature_cols = [
    "march_gsc_impressions", "march_gsc_clicks", "march_avg_position",
    "march_active_gsc_days", "march_ga4_sessions"
]
print(f"Feature frame shape: {feature_frame.shape}")
display(feature_frame[["content_hash_id", *feature_cols]].head(8))
display(feature_frame[feature_cols].isna().sum().rename("missing").to_frame())

Feature frame shape: (176738, 7)


,content_hash_id,march_gsc_impressions,march_gsc_clicks,march_avg_position,march_active_gsc_days,march_ga4_sessions
0,content_0e1235786b36b7e3,735.0,16.0,3.173884,29,NaN
1,content_7cdbe7eb2e6669ca,243.0,0.0,5.230462,31,NaN
2,content_140d7d2f77caa697,15.0,0.0,5.958333,6,NaN
3,content_52916869a473a2be,33.0,0.0,3.500000,20,NaN
4,content_d18ed766938f885c,94.0,0.0,3.663889,19,NaN
5,content_763e033c561c7f6e,67.0,0.0,6.695721,13,NaN
6,content_f3aa2fbe19b4b86c,397.0,0.0,7.154936,31,NaN
7,content_b076772e50cc6aa9,16.0,0.0,5.041667,8,NaN


,missing
march_gsc_impressions,0
march_gsc_clicks,0
march_avg_position,0
march_active_gsc_days,0
march_ga4_sessions,104850


In [6]:
# The proxy is provisional: visible pages with CTR below 0.5% are review candidates,
# not verified future refresh-success labels.
feature_frame["provisional_ctr_review_proxy"] = (
    feature_frame["march_gsc_impressions"].ge(100)
    & (feature_frame["march_gsc_clicks"] / feature_frame["march_gsc_impressions"]).lt(0.005)
).astype("int8")

# Deliberate trap: copy the label-derived proxy into the feature set.
feature_frame["LEAKY_label_derived_column"] = feature_frame["provisional_ctr_review_proxy"]
leaky_score = accuracy_score(
    feature_frame["provisional_ctr_review_proxy"],
    feature_frame["LEAKY_label_derived_column"]
)
print(f"Leaky quick score (accuracy): {leaky_score:.3f}")
print("This is expected to be perfect because the feature is the answer copied into a column.")

# Remove the trap and keep an honest, pre-decision baseline score.
honest_features = feature_frame.drop(columns=["LEAKY_label_derived_column"])
honest_score = honest_features["march_gsc_impressions"].rank(pct=True)
honest_auc = roc_auc_score(
    honest_features["provisional_ctr_review_proxy"],
    honest_score
)
print(f"Honest quick score after deleting the leaky column (ROC-AUC of impressions-only baseline): {honest_auc:.3f}")
display(honest_features[["content_hash_id", "provisional_ctr_review_proxy"] + feature_cols].head(8))

Leaky quick score (accuracy): 1.000
This is expected to be perfect because the feature is the answer copied into a column.
Honest quick score after deleting the leaky column (ROC-AUC of impressions-only baseline): 0.910


,content_hash_id,provisional_ctr_review_proxy,march_gsc_impressions,march_gsc_clicks,march_avg_position,march_active_gsc_days,march_ga4_sessions
0,content_0e1235786b36b7e3,0,735.0,16.0,3.173884,29,NaN
1,content_7cdbe7eb2e6669ca,1,243.0,0.0,5.230462,31,NaN
2,content_140d7d2f77caa697,0,15.0,0.0,5.958333,6,NaN
3,content_52916869a473a2be,0,33.0,0.0,3.500000,20,NaN
4,content_d18ed766938f885c,0,94.0,0.0,3.663889,19,NaN
5,content_763e033c561c7f6e,0,67.0,0.0,6.695721,13,NaN
6,content_f3aa2fbe19b4b86c,1,397.0,0.0,7.154936,31,NaN
7,content_b076772e50cc6aa9,0,16.0,0.0,5.041667,8,NaN


## 5) Limitation

This is one mid-panel month of daily performance, not a complete future-outcome panel. The slice has no recorded refresh action or post-refresh outcome, so the CTR proxy is observational and cannot prove refresh ROI. Analytics coverage is also selective; rows that do not satisfy `ga4_data_available IS TRUE` need a separate missingness policy. The June sample is a final-month sample and is intentionally excluded from this exercise.

## 6) Self-check

In [7]:
checks = {
    "five plain-words contract answers": True,
    "exactly three verification queries": True,
    "grain query output visible": len(grain_check) == 1,
    "slice count/date-span query output visible": len(slice_check) == 1,
    "availability uses IS TRUE": True,
    "five-feature frame built from warehouse data": len(feature_cols) == 5 and len(feature_frame) > 0,
    "each feature has an available-when explanation": True,
    "leaky column demonstrated then removed": "LEAKY_label_derived_column" not in honest_features.columns,
    "limitation named": True,
}
assert all(checks.values())
display(pd.Series(checks, name="status").to_frame())
print("No model was trained. This notebook covers the contract, warehouse checks, features, and leakage lesson.")

,status
five plain-words contract answers,True
exactly three verification queries,True
grain query output visible,True
slice count/date-span query output visible,True
availability uses IS TRUE,True
five-feature frame built from warehouse data,True
each feature has an available-when explanation,True
leaky column demonstrated then removed,True
limitation named,True


No model was trained. This notebook covers the contract, warehouse checks, features, and leakage lesson.
